In [1]:
import pandas as pd
import re

In [7]:
#helper function to clean song names
def normalize_song_name(name):
    if pd.isna(name):
        return name
    name = name.replace("’", "'") 
    name = re.sub(r'\bAnd\b', '&', name)
    return name.strip()

In [3]:
df = pd.read_csv("../data/the_weeknd_setlists.csv", parse_dates=["event_date"])
tour_name = "After Hours Til Dawn"

In [4]:
#build df containing info for each show
tour_df = df[df["tour"] == tour_name].copy()
shows = tour_df[["event_date", "venue", "city", "country"]].drop_duplicates().sort_values("event_date").reset_index(drop=True)
shows["show_index"] = range(len(shows))

In [8]:
#find song_play_rate to narrow down pool of rotating songs
tour_df["song_name_clean"] = tour_df["song_name"].apply(normalize_song_name)
total_shows = tour_df.groupby(["event_date", "venue"]).ngroups
song_show_counts = tour_df.groupby("song_name")["event_date"].nunique()
song_play_rate = (song_show_counts / total_shows).sort_values(ascending=False)
rotating_pool = song_play_rate[(song_play_rate >= 0.05) & (song_play_rate <= 0.85)].index.tolist()

In [10]:

#build df of whether song was played at each show
rows = []
played_lookup = tour_df.groupby(["event_date", "venue"])["song_name"].apply(set).to_dict()

for _, show in shows.iterrows():
    played_set = played_lookup.get((show["event_date"], show["venue"]), set())
    for song in rotating_pool:
        rows.append({
            "event_date": show["event_date"],
            "venue": show["venue"],
            "city": show["city"],
            "country": show["country"],
            "show_index": show["show_index"],
            "song_name": song,
            "was_played": int(song in played_set)
        })

model_df = pd.DataFrame(rows)
print(model_df)

     event_date                    venue          city        country  \
0    2022-07-14  Lincoln Financial Field  Philadelphia  United States   
1    2022-07-14  Lincoln Financial Field  Philadelphia  United States   
2    2022-07-14  Lincoln Financial Field  Philadelphia  United States   
3    2022-07-14  Lincoln Financial Field  Philadelphia  United States   
4    2022-07-14  Lincoln Financial Field  Philadelphia  United States   
...         ...                      ...           ...            ...   
6105 2026-07-12          Stade de France   Saint-Denis         France   
6106 2026-07-12          Stade de France   Saint-Denis         France   
6107 2026-07-12          Stade de France   Saint-Denis         France   
6108 2026-07-12          Stade de France   Saint-Denis         France   
6109 2026-07-12          Stade de France   Saint-Denis         France   

      show_index                                   song_name  was_played  
0              0                                

In [ ]:
#test-train split
cutoff_index = int(shows["show_index"].max() * 0.8)

train_df = model_df[model_df["show_index"] <= cutoff_index]
test_df = model_df[model_df["show_index"] > cutoff_index]